In [1]:
import os, re, shutil
import pandas as pd

df = pd.read_csv("../data/inventory_final.csv")
df = df[df["folder_class"] != "No Tumor"]            # tumor classes only

remap = {"0": "0", "1": "1", "3": "2"}               # Glioma, Meningioma, Pituitary
out = "../yolo_data"
shutil.rmtree(out, ignore_errors=True)

def safe(s):
    return re.sub(r"[^A-Za-z0-9_.-]", "_", s)

for _, r in df.iterrows():
    name = safe(f"{r['folder_class']}_{r['stem']}")
    ext = os.path.splitext(r["image_path"])[1]
    img_out = f"{out}/images/{r['split']}"
    lbl_out = f"{out}/labels/{r['split']}"
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    shutil.copy(r["image_path"], f"{img_out}/{name}{ext}")

    lbl_src = r["image_path"].replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    lines = []
    for line in open(lbl_src).read().strip().splitlines():
        parts = line.split()
        lines.append(" ".join([remap[parts[0]]] + parts[1:]))
    open(f"{lbl_out}/{name}.txt", "w").write("\n".join(lines))

with open(f"{out}/data.yaml", "w") as f:
    f.write("path: .\ntrain: images/train\nval: images/val\ntest: images/test\n")
    f.write("names:\n  0: Glioma\n  1: Meningioma\n  2: Pituitary\n")

# checks
for split in ["train", "val", "test"]:
    n_img = len(os.listdir(f"{out}/images/{split}"))
    n_lbl = len(os.listdir(f"{out}/labels/{split}"))
    print(split, "-> images:", n_img, "| labels:", n_lbl)

ids = set()
for split in ["train", "val", "test"]:
    for f in os.listdir(f"{out}/labels/{split}"):
        ids |= {l.split()[0] for l in open(f"{out}/labels/{split}/{f}").read().splitlines() if l}
print("class ids used:", ids)

shutil.make_archive("../yolo_data", "zip", out)       # zip to upload on Kaggle
print("zip ready: ../yolo_data.zip")

train -> images: 2461 | labels: 2461
val -> images: 405 | labels: 405
test -> images: 435 | labels: 435
class ids used: {'2', '0', '1'}
zip ready: ../yolo_data.zip
